# Parameter Explorer

Edit the next cell to test a new double-slit or PML configuration. The core
Hamiltonian code does not need to be modified. Set `USE_PML = False` to run a
periodic box without an absorbing layer.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from double_slit_pml.model import (
    Geometry, PMLSettings, PlaneWaveModel, compact_packet,
    evolve, project_separable_state, reconstruct,
)

## Parameters — edit this cell

In [ ]:
# Double-slit geometry
SLIT_WIDTH = 0.30
SLIT_SEPARATION = 0.40
BARRIER_THICKNESS = 0.20
BARRIER_HEIGHT = 1.0

# PML and box
USE_PML = True
PML_START = 1.50
PML_THICKNESS = 2.00
PML_ORDER = 4
TARGET_REFLECTION = 1e-3
LY = 6.0
NX = 80
NY = 30

# Incident packet and time interval
PACKET_LEFT = -1.0
PACKET_RIGHT = -0.525
K0 = 30.0
T_FINAL = 0.20
NUMBER_OF_SNAPSHOTS = 21

# Visualization
NUMBER_OF_PLOTTED_SNAPSHOTS = 9
PLOT_COLUMNS = 3
DENSITY_NORMALIZATION = 'physical'  # 'physical', 'integral', or 'absolute'
COLORMAP = 'turbo'
COLOR_GAMMA = 0.5  # below 1 emphasizes low densities; 1 is linear
VIEW_X_MIN, VIEW_X_MAX = -1.0, 1.45
VIEW_Y_MIN, VIEW_Y_MAX = -1.5, 1.5

In [ ]:
geometry = Geometry(
    slit_width=SLIT_WIDTH,
    slit_separation=SLIT_SEPARATION,
    barrier_thickness=BARRIER_THICKNESS,
    barrier_height=BARRIER_HEIGHT,
)
pml = PMLSettings(
    start=PML_START,
    thickness=PML_THICKNESS,
    order=PML_ORDER,
    target_reflection=TARGET_REFLECTION,
)
model = PlaneWaveModel(
    Lx=pml.outer_half_length,
    Ly=LY,
    nx=NX,
    ny=NY,
    geometry=geometry,
    pml=pml if USE_PML else None,
)
psi0 = project_separable_state(
    model,
    lambda x: compact_packet(x, left=PACKET_LEFT, right=PACKET_RIGHT, k0=K0),
)
times = np.linspace(0.0, T_FINAL, NUMBER_OF_SNAPSHOTS)
states = evolve(model, psi0, times)
probability = np.sum(np.abs(states)**2, axis=1)
probability[-1] / probability[0]

In [ ]:
x = np.linspace(VIEW_X_MIN, VIEW_X_MAX, 300)
y = np.linspace(VIEW_Y_MIN, VIEW_Y_MAX, 190)
indices = np.unique(np.rint(np.linspace(
    0, len(times) - 1, NUMBER_OF_PLOTTED_SNAPSHOTS
)).astype(int))

raw_densities = [
    np.abs(reconstruct(model, states[index], x, y))**2
    for index in indices
]
physical_x = np.abs(x) <= PML_START if USE_PML else np.ones_like(x, dtype=bool)
physical_probability = np.asarray([
    np.trapezoid(np.trapezoid(density[physical_x, :], y, axis=1), x[physical_x])
    for density in raw_densities
])
if physical_probability[0] <= np.finfo(float).eps:
    raise ValueError('The initial state has no probability in the displayed physical region')
physical_probability_ratio = physical_probability / physical_probability[0]
physical_peak_density = np.asarray([
    np.max(density[physical_x, :]) for density in raw_densities
])

if DENSITY_NORMALIZATION == 'physical':
    floor = max(np.finfo(float).eps, 1e-12 * physical_peak_density[0])
    densities = [
        density / peak if peak > floor else np.zeros_like(density)
        for density, peak in zip(raw_densities, physical_peak_density)
    ]
    colorbar_label = r'$|\psi|^2/\rho_{\max,\rm phys}(t)$'
    vmax = 1.0
elif DENSITY_NORMALIZATION == 'integral':
    floor = max(np.finfo(float).eps, 1e-12 * physical_probability[0])
    densities = [
        density / p_phys if p_phys > floor else np.zeros_like(density)
        for density, p_phys in zip(raw_densities, physical_probability)
    ]
    colorbar_label = r'$|\psi|^2/P_{\rm phys}(t)$'
    vmax = max(np.quantile(density, 0.995) for density in densities)
elif DENSITY_NORMALIZATION == 'absolute':
    densities = raw_densities
    colorbar_label = r'$|\psi|^2$'
    vmax = max(np.quantile(density, 0.995) for density in densities)
else:
    raise ValueError("DENSITY_NORMALIZATION must be 'physical', 'integral', or 'absolute'")

if COLOR_GAMMA <= 0:
    raise ValueError('COLOR_GAMMA must be positive')
color_norm = PowerNorm(gamma=COLOR_GAMMA, vmin=0.0, vmax=vmax)

columns = min(PLOT_COLUMNS, len(indices))
rows = int(np.ceil(len(indices) / columns))
fig, axes = plt.subplots(
    rows, columns, figsize=(3.8 * columns, 3.2 * rows),
    sharex=True, sharey=True, squeeze=False, constrained_layout=True,
)
image = None
for position, (ax, index, density) in enumerate(zip(axes.flat, indices, densities)):
    image = ax.imshow(
        density.T, origin='lower', extent=(x.min(), x.max(), y.min(), y.max()),
        aspect='auto', cmap=COLORMAP, norm=color_norm,
    )
    ax.set_title(
        rf'$t={times[index]:.3f}$   '
        rf'$\rho_{{\max,\rm phys}}={physical_peak_density[position]:.3g}$',
        fontsize=10,
    )
    ax.set_xlabel('x')
    if position % columns == 0:
        ax.set_ylabel('y')
for ax in axes.flat[len(indices):]:
    ax.axis('off')
fig.colorbar(image, ax=axes.ravel().tolist(), label=colorbar_label, shrink=0.85)

plt.figure(figsize=(7, 3))
plt.plot(times, probability / probability[0], label=r'$P_{\rm total}(t)/P_{\rm total}(0)$')
plt.plot(
    times[indices], physical_probability_ratio, marker='o',
    label=r'$P_{\rm phys}(t)/P_{\rm phys}(0)$',
)
plt.xlabel('time')
plt.ylabel('retained probability')
plt.legend()
plt.tight_layout();

For automated parameter studies, use `scripts/run_experiment.py`.
Run `python scripts/run_experiment.py --help` to list all command-line options.